In [1]:
import pandas as pd

df = pd.read_csv('train.csv')


df['full_text'] = df['Title'] + " " + df['Description']

print(f"Dataset Shape: {df.shape}")
print(df[['full_text', 'Class Index']].head())

Dataset Shape: (120000, 4)
                                           full_text  Class Index
0  Wall St. Bears Claw Back Into the Black (Reute...            3
1  Carlyle Looks Toward Commercial Aerospace (Reu...            3
2  Oil and Economy Cloud Stocks' Outlook (Reuters...            3
3  Iraq Halts Oil Exports from Main Southern Pipe...            3
4  Oil prices soar to all-time record, posing new...            3


In [ ]:
from sklearn.model_selection import train_test_split

X = df['full_text'].values

y = df['Class Index'].values - 1


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

vocab_size = 10000  
max_length = 100    
trunc_type = 'post'
padding_type = 'post'
oov_tok = "<OOV>"   

tokenizer = Tokenizer(num_words=vocab_size, oov_token=oov_tok)
tokenizer.fit_on_texts(X_train)

train_sequences = tokenizer.texts_to_sequences(X_train)
train_padded = pad_sequences(train_sequences, maxlen=max_length, padding=padding_type, truncating=trunc_type)

test_sequences = tokenizer.texts_to_sequences(X_test)
test_padded = pad_sequences(test_sequences, maxlen=max_length, padding=padding_type, truncating=trunc_type)

c:\Users\WALTON\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GlobalAveragePooling1D, Dense, Dropout

model = Sequential([

    Embedding(vocab_size, 16, input_length=max_length),
    GlobalAveragePooling1D(),
    Dense(24, activation='relu'),
    Dropout(0.2),
    Dense(4, activation='softmax') 
])

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

history = model.fit(
    train_padded, y_train, 
    epochs=10, 
    validation_data=(test_padded, y_test),
    verbose=1
)

c:\Users\WALTON\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/10
3000/3000 ━━━━━━━━━━━━━━━━━━━━ 18s 4ms/step - accuracy: 0.7995 - loss: 0.5684 - val_accuracy: 0.9029 - val_loss: 0.3056
Epoch 2/10
3000/3000 ━━━━━━━━━━━━━━━━━━━━ 10s 3ms/step - accuracy: 0.9098 - loss: 0.2854 - val_accuracy: 0.9144 - val_loss: 0.2628
Epoch 3/10
3000/3000 ━━━━━━━━━━━━━━━━━━━━ 10s 3ms/step - accuracy: 0.9196 - loss: 0.2452 - val_accuracy: 0.9140 - val_loss: 0.2558
Epoch 4/10
3000/3000 ━━━━━━━━━━━━━━━━━━━━ 10s 3ms/step - accuracy: 0.9271 - loss: 0.2223 - val_accuracy: 0.9132 - val_loss: 0.2598
Epoch 5/10
3000/3000 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step - accuracy: 0.9320 - loss: 0.2054 - val_accuracy: 0.9135 - val_loss: 0.2569
Epoch 6/10
3000/3000 ━━━━━━━━━━━━━━━━━━━━ 10s 3ms/step - accuracy: 0.9362 - loss: 0.1920 - val_accuracy: 0.9106 - val_loss: 0.2664
Epoch 7/10
3000/3000 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step - accuracy: 0.9393 - loss: 0.1812 - val_accuracy: 0.9018 - val_loss: 0.2948
Epoch 8/10
3000/3000 ━━━━━━━━━━━━━━━━━━━━ 10s 3ms/step - accuracy: 0.9419 - loss: 0.1

In [ ]:
loss, accuracy = model.evaluate(test_padded, y_test)
print(f"Test Accuracy: {accuracy*100:.2f}%")

750/750 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9124 - loss: 0.2832
Test Accuracy: 91.24%
